# 02 — Preprocessing and Feature Engineering
**CMD:** `jupyter lab` → **Run All**; alternative terminal commands: `python -m src.preprocessing` and `python -m src.feature_engineering`.

**Purpose:** generate cleaned weekly records, 4 blood-type rows per date, historical lag/rolling features, and next-week targets.

In [ ]:
# CMD / purpose: Run this cell first in each notebook.
# Makes imports work whether Jupyter starts from the project root or notebooks/.
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open Jupyter from the blood-shortage-ml project directory.')
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

In [ ]:
# CMD / purpose: generate cleaned_weekly.csv (optional intermediate).
from src.preprocessing import save_cleaned
clean = save_cleaned()
print('Clean dates:', len(clean), 'Remaining missing cells:', clean.isna().sum().sum())
display(clean.head(3))

In [ ]:
# CMD / purpose: create data/processed_data.csv for model training.
from src.feature_engineering import generate_processed, FEATURE_COLUMNS
processed = generate_processed()
print('Feature rows:', processed.shape[0])
print('Feature columns:', len(FEATURE_COLUMNS))
print('Next-week labeled examples:', processed.next_risk.notna().sum())
display(processed[['date','blood_type','demand','supply','coverage','observed_risk','target_date','next_demand','next_risk']].head(10))

In [ ]:
# CMD / purpose: verify risk class distribution and the set of model input features.
print('Class counts:\n', processed.next_risk.value_counts(dropna=False))
print('Model input variables (no next_* target columns):')
for col in FEATURE_COLUMNS: print('-', col)
assert all(not col.startswith('next_') for col in FEATURE_COLUMNS)
assert 'target_date' not in FEATURE_COLUMNS
assert processed.groupby('date')['blood_type'].nunique().eq(4).all()

### Target leakage check
At observation week **t**, we use current and earlier information. **`next_risk`** and **`next_demand`** are calculated from the next recorded week and excluded from every model feature. Only pairs separated by 5–9 days are labeled. Missing values are imputed inside a pipeline trained on the training partition only.

In [ ]:
# CMD / purpose: inspect a small example from the same blood type ordered in time.
example = processed.loc[processed.blood_type.eq('A'), ['date','demand','demand_lag1','demand_rolling4','next_demand','target_date']].tail(7)
display(example)